# Tier C: The Transformer
## Parameter-Efficient Fine-Tuning with LoRA for Stylometry Detection

**Objective:** Use state-of-the-art transformer models with PEFT (LoRA) for Human vs AI text classification

**Architecture:**
- **Base Model**: DistilBERT (distilbert-base-uncased)
- **Fine-Tuning Method**: LoRA (Low-Rank Adaptation)
- **Task**: Binary sequence classification

---

## 🔗 Building on Task 1 & Task 2 (Tiers A & B)

**Evolution of Approaches:**

### **Task 1: The Explorer** (Exploratory Data Analysis)
- **Focus**: Mathematical proof of distinction
- **Findings**: TTR variance, structural drift, POS entropy differentiate Human/AI
- **Result**: Established statistical baselines

### **Task 2.1 (Tier A): The Statistician** (Traditional ML)
- **Focus**: Hand-crafted features + XGBoost
- **Method**: Engineer features based on Task 1 discoveries
- **Strength**: Interpretable, fast, baseline performance

### **Task 2.2 (Tier B): The Semanticist** (Neural + Embeddings)
- **Focus**: Pre-trained semantics + feedforward network
- **Method**: GloVe embeddings capture word meaning
- **Strength**: Semantic relationships, better than pure statistics

### **Task 2.3 (Tier C): The Transformer** (SOTA + PEFT)
- **Focus**: Contextual embeddings + self-attention
- **Method**: DistilBERT with LoRA fine-tuning
- **Strength**: Captures long-range dependencies, contextual understanding
- **Expected**: Best performance, learns patterns Task 1 discovered + more

**Why This Progression Matters:**
1. Task 1 proved mathematical distinction exists
2. Tier A validated we can classify with interpretable features
3. Tier B showed semantic information helps
4. Tier C tests if transformers can **automatically discover** what Task 1 found manually

---

**Key Features:**
- **Parameter Efficient**: Train only ~0.6% of model parameters
- **LoRA Configuration**: r=8, alpha=16, dropout=0.1
- **Target Modules**: Query and Value attention layers
- **Evaluation**: Overall accuracy + Mimic subset analysis

---

**Why LoRA?**
- Reduces trainable parameters by 99%+
- Maintains performance comparable to full fine-tuning
- Faster training and lower memory requirements
- Enables fine-tuning large models on consumer GPUs
- **Can learn patterns from Task 1 without explicit feature engineering**

## 🚨 IMPORTANT: Google Drive Setup (Run This First in Colab!)

**Why?** Colab's storage is temporary and gets deleted when your session ends. Save to Google Drive to keep your trained model permanently!

## 1. Setup and Installation

In [33]:
# Install required packages
!pip install transformers datasets peft accelerate scikit-learn pandas numpy torch -q

print("✅ All packages installed successfully!")

✅ All packages installed successfully!


In [34]:
# Import libraries
import pandas as pd
import numpy as np
from typing import Dict, List
import warnings
warnings.filterwarnings('ignore')

# Hugging Face libraries
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# PEFT (Parameter-Efficient Fine-Tuning)
from peft import LoraConfig, get_peft_model, TaskType

# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set random seeds
import torch
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

✅ Libraries imported successfully!
PyTorch version: 2.8.0+cu128
CUDA available: False


## 2. Configuration

**IMPORTANT:** Update these file paths to match your CSV locations

In [36]:
# CSV file paths
CSV_PATHS = {
    "Class 1 (Human)": {
        "path": "/home/avani/precog/content/precog.csv",
        "text_column": "text",
        "label": "Human"
    },
    "Class 2 (AI)": {
        "path": "/home/avani/precog/content/class_2_pro_vanilla_combined.csv",
        "text_column": "text",
        "label_column": "author_target"
    },
    "Class 3 (AI Mimic)": {
        "path": "/home/avani/precog/content/class_3_pro_combined.csv",
        "text_column": "text",
        "label_column": "author_target"
    }
}

# Model configuration
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 512
NUM_LABELS = 2

# LoRA configuration
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_lin", "v_lin"]  # DistilBERT attention layers

# Training configuration
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
BATCH_SIZE = 16
TEST_SIZE = 0.2

# Output directory
OUTPUT_DIR = "/home/avani/precog/reports/lora_distilbert"

print("✅ Configuration complete!")
print(f"\n📊 Model: {MODEL_NAME}")
print(f"📊 LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"📊 Target modules: {TARGET_MODULES}")
print(f"📊 Training: {NUM_EPOCHS} epochs, LR={LEARNING_RATE}")

✅ Configuration complete!

📊 Model: distilbert-base-uncased
📊 LoRA r=8, alpha=16, dropout=0.1
📊 Target modules: ['q_lin', 'v_lin']
📊 Training: 2 epochs, LR=0.0002


## 3. Load and Prepare Data

## 3A. DIAGNOSTIC: Check CSV Column Names

Let's verify what columns exist in each CSV file before loading.

In [ ]:
# Diagnostic: Check what columns exist in each CSV
print("=" * 70)
print("DIAGNOSTIC: CSV FILE COLUMNS")
print("=" * 70)

for class_name, config in CSV_PATHS.items():
    print(f"\n📂 {class_name}: {config['path']}")
    try:
        df_check = pd.read_csv(config['path'])
        print(f"   Columns: {list(df_check.columns)}")
        print(f"   Rows: {len(df_check)}")
        
        # Show first few values of each column
        print(f"   Sample data:")
        for col in df_check.columns[:5]:  # First 5 columns
            sample_val = str(df_check[col].iloc[0])[:50] if len(df_check) > 0 else "N/A"
            print(f"     {col}: {sample_val}...")
    except Exception as e:
        print(f"   ✗ Error: {e}")

print("\n" + "=" * 70)

In [ ]:
def load_csv_files() -> pd.DataFrame:
    """
    Load and combine all three CSV files.
    
    Returns:
        Combined DataFrame with 'text', 'label', 'author', and optionally 'filename' columns
    """
    print("=" * 70)
    print("LOADING CSV FILES")
    print("=" * 70)
    
    all_dfs = []
    
    for class_name, config in CSV_PATHS.items():
        try:
            print(f"\n📂 Loading {class_name}: {config['path']}")
            df = pd.read_csv(config['path'])
            
            print(f"   Available columns: {list(df.columns)}")
            
            # Set label
            if 'label_column' in config and config['label_column'] in df.columns:
                df['label'] = df[config['label_column']]
            else:
                df['label'] = config.get('label', 'Unknown')
            
            # Extract author information (for mimicry analysis)
            # Try multiple possible column names
            author_found = False
            
            # Priority 1: Check for 'author' column
            if 'author' in df.columns:
                df['author'] = df['author']
                author_found = True
                print(f"   ✓ Using 'author' column")
            
            # Priority 2: Check for 'author_target' column (for AI datasets)
            elif 'author_target' in df.columns:
                df['author'] = df['author_target']
                author_found = True
                print(f"   ✓ Using 'author_target' column")
            
            # Priority 3: Check for label_column (might contain author info)
            elif 'label_column' in config and config['label_column'] in df.columns:
                df['author'] = df[config['label_column']]
                author_found = True
                print(f"   ✓ Using '{config['label_column']}' column as author")
            
            # Fallback: Set to 'Unknown' (will extract from filename later if available)
            if not author_found:
                df['author'] = 'Unknown'
                print(f"   ⚠️  No author column found, setting to 'Unknown'")
                if 'filename' in df.columns:
                    print(f"   💡 Will extract author from 'filename' column later")
            
            # Keep text, label, author, and filename (if it exists)
            columns_to_keep = [config['text_column'], 'label', 'author']
            if 'filename' in df.columns:
                columns_to_keep.append('filename')
            
            df = df[columns_to_keep].copy()
            
            # Rename text column to 'text'
            df = df.rename(columns={config['text_column']: 'text'})
            
            print(f"   ✓ Loaded {len(df)} samples")
            print(f"   ✓ Kept columns: {list(df.columns)}")
            
            all_dfs.append(df)
            
        except FileNotFoundError:
            print(f"   ✗ File not found: {config['path']}")
        except Exception as e:
            print(f"   ✗ Error: {e}")
            import traceback
            traceback.print_exc()
    
    if not all_dfs:
        raise FileNotFoundError("No CSV files could be loaded.")
    
    df_combined = pd.concat(all_dfs, ignore_index=True)
    
    print(f"\n{'=' * 70}")
    print(f"COMBINED DATASET: {len(df_combined)} total samples")
    print(f"{'=' * 70}")
    print(f"\nLabel distribution:")
    print(df_combined['label'].value_counts())
    print(f"\nAuthor distribution (before filename extraction):")
    print(df_combined['author'].value_counts())
    
    return df_combined


# Load data
df = load_csv_files()

LOADING CSV FILES

📂 Loading Class 1 (Human): /home/avani/precog/content/precog.csv
   ✓ Loaded 2492 samples

📂 Loading Class 2 (AI): /home/avani/precog/content/class_2_pro_vanilla_combined.csv
   ✓ Loaded 500 samples

📂 Loading Class 3 (AI Mimic): /home/avani/precog/content/class_3_pro_combined.csv
   ✓ Loaded 500 samples

COMBINED DATASET: 3492 total samples

Label distribution:
label
Human                     2492
AI_Generic                 500
Robert Louis Stevenson     250
Arthur Conan Doyle         250
Name: count, dtype: int64


In [38]:
# Create binary labels
print("\n=" * 70)
print("CREATING BINARY LABELS")
print("=" * 70)

df['binary_label'] = df['label'].apply(lambda x: 0 if x.lower() == 'human' else 1)

print(f"\nBinary label distribution:")
print(f"  Human (0): {(df['binary_label'] == 0).sum()} samples")
print(f"  AI (1):    {(df['binary_label'] == 1).sum()} samples")

# Display sample
print(f"\n📝 Sample data:")
print(df[['text', 'label', 'binary_label']].head(3))


=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
CREATING BINARY LABELS

Binary label distribution:
  Human (0): 2492 samples
  AI (1):    1000 samples

📝 Sample data:
                                                text  label  binary_label
0  The Strange Case of Dr. Jekyll and Mr. Hyde by...  Human             0
1  Mr. Utterson the lawyer was a man of a rugged ...  Human             0
2  No doubt the feat was easy to Mr. Utterson; fo...  Human             0


## 3B. CRITICAL FIX: Extract Author from Filename

For precog.csv, the author info is in the filename column, not a separate author column!

In [ ]:
# CRITICAL FIX: Extract author from filename in precog.csv
print("=" * 70)
print("FIXING AUTHOR EXTRACTION")
print("=" * 70)

# Function to extract author from filename
def extract_author_from_filename(filename):
    """
    Extract author from filename like 'SMART_Stevenson_Gothic_42.txt'
    Returns 'Robert Louis Stevenson' or 'Arthur Conan Doyle' or 'Unknown'
    """
    if pd.isna(filename):
        return 'Unknown'
    
    filename_str = str(filename).lower()
    
    if 'stevenson' in filename_str:
        return 'Robert Louis Stevenson'
    elif 'doyle' in filename_str:
        return 'Arthur Conan Doyle'
    else:
        return 'Unknown'

# Check if df has 'filename' column (from precog.csv)
if 'filename' in df.columns:
    print("\n✅ Found 'filename' column - extracting authors...")
    
    # Show sample before
    print(f"\n📋 BEFORE:")
    print(f"Sample authors: {df['author'].unique()[:5]}")
    
    # Extract author from filename for rows where author is 'Unknown'
    mask_unknown = df['author'] == 'Unknown'
    if mask_unknown.sum() > 0:
        df.loc[mask_unknown, 'author'] = df.loc[mask_unknown, 'filename'].apply(extract_author_from_filename)
        
        print(f"\n✅ Updated {mask_unknown.sum()} rows")
        print(f"\n📋 AFTER:")
        print(f"Author distribution:\n{df['author'].value_counts()}")
    else:
        print("\n⚠️  No 'Unknown' authors found to update")
else:
    print("\n⚠️  No 'filename' column found - cannot extract authors")
    print(f"   Available columns: {df.columns.tolist()}")

print("\n" + "=" * 70)

## 4. Convert to Hugging Face Dataset

In [ ]:
from sklearn.model_selection import train_test_split

print("=" * 70)
print("CREATING HUGGING FACE DATASET")
print("=" * 70)

# Train-test split (stratified on binary label)
train_df, test_df = train_test_split(
    df[['text', 'label', 'binary_label', 'author']], 
    test_size=TEST_SIZE, 
    random_state=SEED, 
    stratify=df['binary_label']
)

print(f"\n✓ Train set: {len(train_df)} samples")
print(f"✓ Test set:  {len(test_df)} samples")

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df[['text', 'binary_label']].rename(columns={'binary_label': 'labels'}))
test_dataset = Dataset.from_pandas(test_df[['text', 'binary_label']].rename(columns={'binary_label': 'labels'}))

# Store original labels and authors for evaluation
test_labels_original = test_df['label'].values
test_authors = test_df['author'].values
test_binary_labels = test_df['binary_label'].values

dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

print(f"\n✅ Hugging Face Dataset created:")
print(dataset)
print(f"\n📊 Test set composition:")
print(f"  Human samples: {(test_binary_labels == 0).sum()}")
print(f"  AI samples: {(test_binary_labels == 1).sum()}")

CREATING HUGGING FACE DATASET

✓ Train set: 2793 samples
✓ Test set:  699 samples

✅ Hugging Face Dataset created:
DatasetDict({
    train: Dataset({
        features: ['text', 'labels', '__index_level_0__'],
        num_rows: 2793
    })
    test: Dataset({
        features: ['text', 'labels', '__index_level_0__'],
        num_rows: 699
    })
})


## 5. Load Tokenizer and Tokenize Data

In [40]:
print("=" * 70)
print("LOADING TOKENIZER")
print("=" * 70)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"\n✓ Loaded tokenizer: {MODEL_NAME}")
print(f"  Vocabulary size: {len(tokenizer)}")
print(f"  Max length: {MAX_LENGTH}")

# Test tokenizer
test_text = "The quick brown fox jumps over the lazy dog."
test_tokens = tokenizer(test_text, truncation=True, max_length=MAX_LENGTH)
print(f"\n📝 Test tokenization:")
print(f"  Text: {test_text}")
print(f"  Tokens: {tokenizer.convert_ids_to_tokens(test_tokens['input_ids'][:10])}...")
print(f"  Token IDs: {test_tokens['input_ids'][:10]}...")

LOADING TOKENIZER

✓ Loaded tokenizer: distilbert-base-uncased
  Vocabulary size: 30522
  Max length: 512

📝 Test tokenization:
  Text: The quick brown fox jumps over the lazy dog.
  Tokens: ['[CLS]', 'the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']...
  Token IDs: [101, 1996, 4248, 2829, 4419, 14523, 2058, 1996, 13971, 3899]...

✓ Loaded tokenizer: distilbert-base-uncased
  Vocabulary size: 30522
  Max length: 512

📝 Test tokenization:
  Text: The quick brown fox jumps over the lazy dog.
  Tokens: ['[CLS]', 'the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']...
  Token IDs: [101, 1996, 4248, 2829, 4419, 14523, 2058, 1996, 13971, 3899]...


In [41]:
def tokenize_function(examples):
    """
    Tokenize text data.
    """
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False  # Will pad dynamically during training
    )


print("\n=" * 70)
print("TOKENIZING DATASET")
print("=" * 70)
print("\n⚙️  Tokenizing texts (this may take a moment...)\n")

# Tokenize datasets
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],
    desc="Tokenizing"
)

print(f"\n✅ Tokenization complete:")
print(tokenized_datasets)

# Show sample
print(f"\n📝 Sample tokenized data:")
print(f"  Keys: {tokenized_datasets['train'][0].keys()}")
print(f"  Input IDs length: {len(tokenized_datasets['train'][0]['input_ids'])}")
print(f"  Label: {tokenized_datasets['train'][0]['labels']}")


=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
TOKENIZING DATASET

⚙️  Tokenizing texts (this may take a moment...)



Tokenizing:   0%|          | 0/2793 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/699 [00:00<?, ? examples/s]


✅ Tokenization complete:
DatasetDict({
    train: Dataset({
        features: ['labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2793
    })
    test: Dataset({
        features: ['labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 699
    })
})

📝 Sample tokenized data:
  Keys: dict_keys(['labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'])
  Input IDs length: 155
  Label: 0


In [42]:
# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("✅ Data collator created for dynamic padding")

✅ Data collator created for dynamic padding


## 6. Load Base Model

In [43]:
print("=" * 70)
print("LOADING BASE MODEL")
print("=" * 70)

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={0: "Human", 1: "AI"},
    label2id={"Human": 0, "AI": 1}
)

print(f"\n✓ Loaded model: {MODEL_NAME}")
print(f"  Task: Sequence Classification")
print(f"  Number of labels: {NUM_LABELS}")

# Count parameters before LoRA
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model parameters (BEFORE LoRA):")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

LOADING BASE MODEL


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



✓ Loaded model: distilbert-base-uncased
  Task: Sequence Classification
  Number of labels: 2

📊 Model parameters (BEFORE LoRA):
  Total parameters: 66,955,010
  Trainable parameters: 66,955,010


## 7. Apply LoRA (Parameter-Efficient Fine-Tuning)

**LoRA Configuration:**
- **r (rank)**: 8 - Dimension of low-rank matrices
- **alpha**: 16 - Scaling factor for LoRA updates
- **dropout**: 0.1 - Regularization
- **target_modules**: ["q_lin", "v_lin"] - Apply LoRA to Query and Value attention layers only

This reduces trainable parameters from ~67M to ~0.4M (~0.6%)

In [44]:
print("=" * 70)
print("APPLYING LoRA (LOW-RANK ADAPTATION)")
print("=" * 70)

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    inference_mode=False
)

print(f"\n⚙️  LoRA Configuration:")
print(f"  r (rank): {LORA_R}")
print(f"  alpha: {LORA_ALPHA}")
print(f"  dropout: {LORA_DROPOUT}")
print(f"  target_modules: {TARGET_MODULES}")

# Apply LoRA to model
model = get_peft_model(model, lora_config)

print(f"\n✅ LoRA applied successfully!")

APPLYING LoRA (LOW-RANK ADAPTATION)

⚙️  LoRA Configuration:
  r (rank): 8
  alpha: 16
  dropout: 0.1
  target_modules: ['q_lin', 'v_lin']

✅ LoRA applied successfully!


In [45]:
# Print trainable parameters (AFTER LoRA)
print("\n" + "=" * 70)
print("EFFICIENCY CHECK: TRAINABLE PARAMETERS")
print("=" * 70)

model.print_trainable_parameters()

# Calculate manually for clarity
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable_percentage = 100 * trainable_params / total_params

print(f"\n📊 Detailed parameter breakdown:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Trainable %: {trainable_percentage:.2f}%")
print(f"\n✨ Training only {trainable_percentage:.2f}% of parameters!")
print(f"   This is {100 - trainable_percentage:.2f}% more efficient than full fine-tuning!")


EFFICIENCY CHECK: TRAINABLE PARAMETERS
trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925

📊 Detailed parameter breakdown:
  Total parameters: 67,694,596
  Trainable parameters: 739,586
  Trainable %: 1.09%

✨ Training only 1.09% of parameters!
   This is 98.91% more efficient than full fine-tuning!


## 8. Define Training Arguments

In [ ]:
print("=" * 70)
print("CONFIGURING TRAINING")
print("=" * 70)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=50,
    eval_strategy="epoch",  # Monitor validation performance each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,  # Higher accuracy is better
    seed=SEED,
    report_to="none",  # Disable wandb/tensorboard for simplicity
    push_to_hub=False,
    # Early stopping parameters
    save_total_limit=2  # Keep only 2 best checkpoints
)

print(f"\n⚙️  Training configuration:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Evaluation: Every epoch")
print(f"  Early stopping: Load best model at end")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"\n✅ Training arguments configured")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


CONFIGURING TRAINING

⚙️  Training configuration:
  Epochs: 2
  Learning rate: 0.0002
  Batch size: 16
  Output directory: /home/avani/precog/reports/lora_distilbert

✅ Training arguments configured


## 9. Define Metrics

In [47]:
def compute_metrics(eval_pred):
    """
    Compute accuracy for evaluation.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy}


print("✅ Metrics function defined")

✅ Metrics function defined


## 10. Initialize Trainer

In [49]:
print("=" * 70)
print("INITIALIZING TRAINER")
print("=" * 70)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("\n✅ Trainer initialized successfully!")
print(f"  Training samples: {len(tokenized_datasets['train'])}")
print(f"  Evaluation samples: {len(tokenized_datasets['test'])}")

INITIALIZING TRAINER

✅ Trainer initialized successfully!
  Training samples: 2793
  Evaluation samples: 699


## 11. Train the Model

⚠️ **This will take several minutes depending on your GPU/CPU.**

In [50]:
print("=" * 70)
print("STARTING TRAINING")
print("=" * 70)
print("\n🚀 Training LoRA-adapted DistilBERT...\n")

# Train
train_result = trainer.train()

print("\n" + "=" * 70)
print("✅ TRAINING COMPLETE!")
print("=" * 70)

# Print training metrics
print(f"\n📊 Training metrics:")
for key, value in train_result.metrics.items():
    print(f"  {key}: {value:.4f}")

STARTING TRAINING

🚀 Training LoRA-adapted DistilBERT...



Epoch,Training Loss,Validation Loss,Accuracy
1,0.032332,0.015277,0.994278
2,0.005318,0.006768,0.997139



✅ TRAINING COMPLETE!

📊 Training metrics:
  train_runtime: 5376.2114
  train_samples_per_second: 1.0390
  train_steps_per_second: 0.0650
  total_flos: 546878304476592.0000
  train_loss: 0.1080
  epoch: 2.0000


## 12. Evaluate on Test Set

In [ ]:
print("=" * 70)
print("SECTION A: OVERALL TEST SET EVALUATION")
print("=" * 70)

# Evaluate on full test set
eval_results = trainer.evaluate()

print(f"\n📊 Overall Test Set Performance:")
print("=" * 70)
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

test_accuracy = eval_results['eval_accuracy']
print(f"\n✨ Overall Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

In [ ]:
# Get predictions for detailed analysis
print("\n🔮 Generating predictions for detailed analysis...\n")

predictions = trainer.predict(tokenized_datasets['test'])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# Get prediction probabilities for confidence scores
y_pred_probs = torch.softmax(torch.tensor(predictions.predictions), dim=1).numpy()

print("✅ Predictions generated")

EVALUATING ON TEST SET



📊 Test Set Performance:
  eval_loss: 0.0068
  eval_accuracy: 0.9971
  eval_runtime: 133.5343
  eval_samples_per_second: 5.2350
  eval_steps_per_second: 0.3300
  epoch: 2.0000

✨ Final Test Accuracy: 0.9971 (99.71%)


## 13. Detailed Evaluation: Classification Metrics

Overall test set performance with detailed metrics.

In [ ]:
print("=" * 70)
print("DETAILED CLASSIFICATION METRICS")
print("=" * 70)

# Classification report
print("\n📋 Classification Report:")
print("=" * 70)
report = classification_report(
    y_true, 
    y_pred, 
    target_names=['Human (0)', 'AI (1)'],
    digits=4,
    output_dict=True
)
print(classification_report(
    y_true, 
    y_pred, 
    target_names=['Human (0)', 'AI (1)'],
    digits=4
))

# Extract key metrics
precision_human = report['Human (0)']['precision']
recall_human = report['Human (0)']['recall']
f1_human = report['Human (0)']['f1-score']

precision_ai = report['AI (1)']['precision']
recall_ai = report['AI (1)']['recall']
f1_ai = report['AI (1)']['f1-score']

DETAILED EVALUATION METRICS

📋 Classification Report:
              precision    recall  f1-score   support

   Human (0)     1.0000    0.9960    0.9980       499
      AI (1)     0.9901    1.0000    0.9950       200

    accuracy                         0.9971       699
   macro avg     0.9950    0.9980    0.9965       699
weighted avg     0.9972    0.9971    0.9971       699



In [ ]:
# Confusion matrix
print("\n🎯 Confusion Matrix:")
print("=" * 70)
cm = confusion_matrix(y_true, y_pred)
print(f"\n                Predicted")
print(f"                Human    AI")
print(f"Actual Human    {cm[0][0]:5d}  {cm[0][1]:5d}")
print(f"       AI       {cm[1][0]:5d}  {cm[1][1]:5d}")

# Calculate additional metrics
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Positive Rate (Human→AI)
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0  # False Negative Rate (AI→Human)

print(f"\n📈 Detailed Metrics:")
print("=" * 70)
print(f"  True Negatives (Human→Human):   {tn:5d}")
print(f"  False Positives (Human→AI):     {fp:5d}  ← Flagged real humans as AI")
print(f"  False Negatives (AI→Human):     {fn:5d}  ← Missed AI detection")
print(f"  True Positives (AI→AI):         {tp:5d}")
print(f"\n  Sensitivity (Recall on AI):     {sensitivity:.4f}  (How many AI texts we catch)")
print(f"  Specificity (Recall on Human):  {specificity:.4f}  (How many humans correctly identified)")
print(f"  False Positive Rate:            {fpr:.4f}  (Prob. of flagging human as AI)")
print(f"  False Negative Rate:            {fnr:.4f}  (Prob. of missing AI)")

# Store metrics for reporting
overall_metrics = {
    'accuracy': test_accuracy,
    'precision_human': precision_human,
    'recall_human': recall_human,
    'f1_human': f1_human,
    'precision_ai': precision_ai,
    'recall_ai': recall_ai,
    'f1_ai': f1_ai,
    'specificity': specificity,
    'sensitivity': sensitivity,
    'false_positive_rate': fpr,
    'false_negative_rate': fnr,
    'tn': int(tn),
    'fp': int(fp),
    'fn': int(fn),
    'tp': int(tp)
}


🎯 Confusion Matrix:

           Predicted
           Human  AI
Actual Human   497     2
       AI        0   200

📈 Additional Metrics:
  True Negatives:  497
  False Positives: 2
  False Negatives: 0
  True Positives:  200

  Sensitivity (Recall): 1.0000
  Specificity:          0.9960


## 14. SECTION B: Class 2 (Generic AI) Evaluation

Evaluate on AI texts that are NOT mimicking specific authors (generic AI generation).

## 14A. Debugging: Understanding the Author Data

Let's first understand what author names exist in our test set.

In [ ]:
# Diagnostic cell to understand author distribution
print("=" * 70)
print("DIAGNOSTIC: AUTHOR DISTRIBUTION IN TEST SET")
print("=" * 70)

# Show all unique authors and their counts
unique_test_authors = np.unique(test_authors)
print(f"\n📊 Total unique authors: {len(unique_test_authors)}")
print(f"\nAll authors in test set:\n")

author_df = pd.DataFrame({
    'author': unique_test_authors,
    'count': [np.sum(test_authors == a) for a in unique_test_authors],
    'label_distribution': [f"H:{np.sum((test_authors == a) & (test_binary_labels == 0))} AI:{np.sum((test_authors == a) & (test_binary_labels == 1))}" for a in unique_test_authors]
})
author_df = author_df.sort_values('count', ascending=False)

print(author_df.to_string(index=False))

# Check which authors contain Doyle or Stevenson
print(f"\n🔍 Authors containing 'Doyle':")
doyle_authors = [a for a in unique_test_authors if 'Doyle' in str(a)]
if doyle_authors:
    for author in doyle_authors:
        count = np.sum(test_authors == author)
        print(f"  - {author}: {count} samples")
else:
    print("  None found")

print(f"\n🔍 Authors containing 'Stevenson':")
stevenson_authors = [a for a in unique_test_authors if 'Stevenson' in str(a)]
if stevenson_authors:
    for author in stevenson_authors:
        count = np.sum(test_authors == author)
        print(f"  - {author}: {count} samples")
else:
    print("  None found")

print("\n" + "=" * 70)

In [ ]:
print("\n" + "=" * 70)
print("SECTION B: CLASS 2 (GENERIC AI) EVALUATION")
print("=" * 70)

# First, let's debug: show what authors we have
print(f"\n🔍 Debug: Unique authors in test set:")
unique_authors = np.unique(test_authors)
for author in unique_authors[:10]:  # Show first 10
    count = (test_authors == author).sum()
    print(f"  {author}: {count} samples")

# Filter for Class 2: AI samples NOT mimicking Doyle or Stevenson
# FIX: np.char.find returns -1 when NOT found, so we check == -1
class2_mask = (test_binary_labels == 1) & \
              (np.char.find(test_authors.astype(str), 'Doyle') == -1) & \
              (np.char.find(test_authors.astype(str), 'Stevenson') == -1)

print(f"\n🔍 Debug: Filter breakdown:")
print(f"  AI samples (label=1): {(test_binary_labels == 1).sum()}")
print(f"  Samples WITHOUT 'Doyle': {(np.char.find(test_authors.astype(str), 'Doyle') == -1).sum()}")
print(f"  Samples WITHOUT 'Stevenson': {(np.char.find(test_authors.astype(str), 'Stevenson') == -1).sum()}")
print(f"  Final Class 2 samples: {class2_mask.sum()}")

if class2_mask.sum() > 0:
    y_class2_true = y_true[class2_mask]
    y_class2_pred = y_pred[class2_mask]
    
    print(f"\n🤖 Found {class2_mask.sum()} generic AI samples in test set")
    
    # Calculate Class 2 accuracy
    class2_accuracy = accuracy_score(y_class2_true, y_class2_pred)
    
    print(f"\n📊 Class 2 (Generic AI) Performance:")
    print("=" * 70)
    print(f"  Accuracy: {class2_accuracy:.4f} ({class2_accuracy*100:.2f}%)")
    print(f"  Correctly classified: {(y_class2_true == y_class2_pred).sum()}/{len(y_class2_true)}")
    
    # Classification report for Class 2
    print(f"\n📋 Class 2 Classification Report:")
    print("=" * 70)
    print(classification_report(
        y_class2_true, 
        y_class2_pred,
        labels=[1],  # Only AI label
        target_names=['AI (1)'],
        digits=4,
        zero_division=0
    ))
    
    # Store metric
    class2_metrics = {
        'class2_accuracy': class2_accuracy,
        'class2_samples': int(class2_mask.sum()),
        'class2_correct': int((y_class2_true == y_class2_pred).sum())
    }
else:
    print("\n⚠️  No generic AI samples found in test set")
    print("   This might mean:")
    print("   1. All AI samples are mimicking Doyle/Stevenson, OR")
    print("   2. The author names don't contain 'Doyle' or 'Stevenson' strings")
    print("\n   Check the author distribution above to verify.")
    class2_metrics = {}

## 15. SECTION C: Class 3 (Mimicry) Evaluation - THE CRITICAL FIX

**This is the fixed evaluation that tests BOTH:**
1. Real human texts from Doyle and Stevenson (label='Human')
2. AI texts mimicking Doyle and Stevenson (label='AI')

**Question:** Can the model distinguish real author text from AI mimicking that author?

In [ ]:
print("\n" + "=" * 70)
print("SECTION C: CLASS 3 (MIMICRY) EVALUATION - FIXED VERSION")
print("=" * 70)
print("\n🎭 Creating BALANCED subset: Real human authors + AI mimicking them\n")

# Filter for Doyle and Stevenson samples (BOTH human and AI)
mimicry_authors = ['Arthur Conan Doyle', 'Robert Louis Stevenson', 'Doyle', 'Stevenson']
mimicry_mask = np.array([
    any(author_name in str(author) for author_name in mimicry_authors)
    for author in test_authors
])

if mimicry_mask.sum() > 0:
    y_mimicry_true = y_true[mimicry_mask]
    y_mimicry_pred = y_pred[mimicry_mask]
    mimicry_labels = test_binary_labels[mimicry_mask]
    mimicry_authors_list = test_authors[mimicry_mask]
    
    print(f"✅ Found {mimicry_mask.sum()} samples related to Doyle/Stevenson in test set")
    
    # Breakdown by true label
    human_count = (mimicry_labels == 0).sum()
    ai_count = (mimicry_labels == 1).sum()
    
    print(f"\n📊 Mimicry Subset Composition:")
    print("=" * 70)
    print(f"  Real human texts (Doyle/Stevenson):  {human_count:5d} samples")
    print(f"  AI mimicking (Doyle/Stevenson):      {ai_count:5d} samples")
    print(f"  Total mimicry subset:                {mimicry_mask.sum():5d} samples")
    
    if human_count == 0:
        print("\n⚠️  WARNING: No real human texts in subset! Evaluation will be biased.")
    if ai_count == 0:
        print("\n⚠️  WARNING: No AI mimicry texts in subset! Evaluation will be biased.")
    
    # Calculate overall mimicry accuracy
    mimicry_accuracy = accuracy_score(y_mimicry_true, y_mimicry_pred)
    
    print(f"\n📊 Overall Mimicry Subset Performance:")
    print("=" * 70)
    print(f"  Mimicry Accuracy: {mimicry_accuracy:.4f} ({mimicry_accuracy*100:.2f}%)")
    
    # Detailed classification report
    print(f"\n📋 Mimicry Classification Report (BOTH Human + AI):")
    print("=" * 70)
    mimicry_report = classification_report(
        y_mimicry_true, 
        y_mimicry_pred,
        labels=[0, 1],
        target_names=['Human (Real Authors)', 'AI (Mimicking)'],
        digits=4,
        zero_division=0,
        output_dict=True
    )
    print(classification_report(
        y_mimicry_true, 
        y_mimicry_pred,
        labels=[0, 1],
        target_names=['Human (Real Authors)', 'AI (Mimicking)'],
        digits=4,
        zero_division=0
    ))
    
    # Confusion matrix for mimicry subset
    print(f"\n🎯 Mimicry Confusion Matrix:")
    print("=" * 70)
    cm_mimicry = confusion_matrix(y_mimicry_true, y_mimicry_pred)
    print(f"\n                       Predicted")
    print(f"                       Human    AI")
    print(f"Actual Human (Real)    {cm_mimicry[0][0]:5d}  {cm_mimicry[0][1]:5d}")
    print(f"       AI (Mimic)      {cm_mimicry[1][0]:5d}  {cm_mimicry[1][1]:5d}")
    
    # Key metrics
    tn_mim, fp_mim, fn_mim, tp_mim = cm_mimicry.ravel()
    
    # Mimicry detection rate (recall on AI class)
    mimicry_detection_rate = tp_mim / (tp_mim + fn_mim) if (tp_mim + fn_mim) > 0 else 0
    
    # Human false positive rate (1 - specificity)
    human_fpr = fp_mim / (fp_mim + tn_mim) if (fp_mim + tn_mim) > 0 else 0
    
    print(f"\n📈 Critical Mimicry Metrics:")
    print("=" * 70)
    print(f"  True Negatives (Real→Human):       {tn_mim:5d}  ← Real authors correctly kept as human")
    print(f"  False Positives (Real→AI):         {fp_mim:5d}  ← Real authors wrongly flagged as AI")
    print(f"  False Negatives (Mimic→Human):     {fn_mim:5d}  ← AI mimics that fooled the model")
    print(f"  True Positives (Mimic→AI):         {tp_mim:5d}  ← AI mimics successfully detected")
    print(f"\n  Mimicry AI Detection Rate:         {mimicry_detection_rate:.4f}  (Recall on AI mimics)")
    print(f"  Human False Positive Rate:         {human_fpr:.4f}  (Real authors flagged as AI)")
    
    print(f"\n💡 Interpretation:")
    print("=" * 70)
    print(f"  Can detect {mimicry_detection_rate*100:.1f}% of AI texts mimicking Doyle/Stevenson")
    print(f"  Incorrectly flags {human_fpr*100:.1f}% of real Doyle/Stevenson as AI")
    
    # Comparison with overall performance
    print(f"\n🔍 Comparison with Overall Test Performance:")
    print("=" * 70)
    print(f"  Overall test accuracy:     {test_accuracy:.4f}")
    print(f"  Mimicry subset accuracy:   {mimicry_accuracy:.4f}")
    print(f"  Difference:                {abs(test_accuracy - mimicry_accuracy):.4f}")
    
    if mimicry_accuracy < test_accuracy - 0.05:
        print(f"\n⚠️  Mimicry detection is SIGNIFICANTLY HARDER!")
        print(f"   Model struggles more with author-specific AI mimics")
    elif mimicry_accuracy > test_accuracy + 0.05:
        print(f"\n✅ Mimicry detection is EASIER!")
        print(f"   Model performs better on author-specific texts")
    else:
        print(f"\n➡️  Mimicry detection is COMPARABLE to overall performance")
    
    # Store metrics
    class3_metrics = {
        'mimicry_accuracy': mimicry_accuracy,
        'mimicry_samples': int(mimicry_mask.sum()),
        'mimicry_human_samples': int(human_count),
        'mimicry_ai_samples': int(ai_count),
        'mimicry_detection_rate': mimicry_detection_rate,
        'human_false_positive_rate': human_fpr,
        'mimicry_tn': int(tn_mim),
        'mimicry_fp': int(fp_mim),
        'mimicry_fn': int(fn_mim),
        'mimicry_tp': int(tp_mim)
    }
    
else:
    print("\n⚠️  No Doyle/Stevenson samples found in test set")
    class3_metrics = {}

## 16. Error Analysis: Misclassified Samples

Analyze where the model made mistakes to understand failure modes.

In [ ]:
print("\n" + "=" * 70)
print("ERROR ANALYSIS: MISCLASSIFIED SAMPLES")
print("=" * 70)

# Get test texts (need to reconstruct from DataFrame)
test_texts = test_df['text'].values

# Find errors
errors = y_true != y_pred

# False Negatives: AI classified as Human
false_negatives = (y_true == 1) & (y_pred == 0)
fn_indices = np.where(false_negatives)[0]

# False Positives: Human classified as AI
false_positives = (y_true == 0) & (y_pred == 1)
fp_indices = np.where(false_positives)[0]

print(f"\n📊 Error Summary:")
print("=" * 70)
print(f"  Total errors: {errors.sum()} / {len(y_true)} ({errors.sum()/len(y_true)*100:.2f}%)")
print(f"  False Negatives (AI→Human): {false_negatives.sum()}")
print(f"  False Positives (Human→AI): {false_positives.sum()}")

In [ ]:
# Analyze False Negatives (AI texts that fooled the model)
print(f"\n🔴 FALSE NEGATIVES: AI texts classified as Human (Model missed these)")
print("=" * 70)

if len(fn_indices) > 0:
    print(f"\nShowing up to 5 examples:\n")
    for i, idx in enumerate(fn_indices[:5]):
        text = test_texts[idx]
        confidence = y_pred_probs[idx][0]  # Confidence in Human (wrong prediction)
        author = test_authors[idx]
        
        print(f"\nExample {i+1}:")
        print(f"  Author: {author}")
        print(f"  True Label: AI (1)")
        print(f"  Predicted: Human (0)")
        print(f"  Confidence in 'Human': {confidence:.4f}")
        print(f"  Text preview: {text[:200]}...")
        print("-" * 70)
else:
    print("\n✅ No false negatives! Model caught all AI texts.")

In [ ]:
# Analyze False Positives (Human texts wrongly flagged as AI)
print(f"\n🟡 FALSE POSITIVES: Human texts classified as AI (False alarms)")
print("=" * 70)

if len(fp_indices) > 0:
    print(f"\nShowing up to 5 examples:\n")
    for i, idx in enumerate(fp_indices[:5]):
        text = test_texts[idx]
        confidence = y_pred_probs[idx][1]  # Confidence in AI (wrong prediction)
        author = test_authors[idx]
        
        print(f"\nExample {i+1}:")
        print(f"  Author: {author}")
        print(f"  True Label: Human (0)")
        print(f"  Predicted: AI (1)")
        print(f"  Confidence in 'AI': {confidence:.4f}")
        print(f"  Text preview: {text[:200]}...")
        print("-" * 70)
else:
    print("\n✅ No false positives! Model never flagged real humans as AI.")

## 17. Final Metrics Summary

Consolidated report of all evaluation metrics.

In [ ]:
print("\n" + "=" * 70)
print("FINAL METRICS SUMMARY")
print("=" * 70)

# Combine all metrics
final_metrics = {
    **overall_metrics,
    **class2_metrics,
    **class3_metrics
}

print("\n📊 SECTION A: OVERALL TEST SET")
print("-" * 70)
print(f"  Test Accuracy:              {final_metrics['accuracy']:.4f}")
print(f"  Precision (Human):          {final_metrics['precision_human']:.4f}")
print(f"  Recall (Human):             {final_metrics['recall_human']:.4f}")
print(f"  F1-Score (Human):           {final_metrics['f1_human']:.4f}")
print(f"  Precision (AI):             {final_metrics['precision_ai']:.4f}")
print(f"  Recall (AI):                {final_metrics['recall_ai']:.4f}")
print(f"  F1-Score (AI):              {final_metrics['f1_ai']:.4f}")
print(f"  Specificity:                {final_metrics['specificity']:.4f}")
print(f"  Sensitivity:                {final_metrics['sensitivity']:.4f}")
print(f"  False Positive Rate:        {final_metrics['false_positive_rate']:.4f}")
print(f"  False Negative Rate:        {final_metrics['false_negative_rate']:.4f}")

if class2_metrics:
    print("\n📊 SECTION B: CLASS 2 (GENERIC AI)")
    print("-" * 70)
    print(f"  Class 2 Samples:            {final_metrics['class2_samples']}")
    print(f"  Class 2 Accuracy:           {final_metrics['class2_accuracy']:.4f}")
    print(f"  Correctly Classified:       {final_metrics['class2_correct']}/{final_metrics['class2_samples']}")

if class3_metrics:
    print("\n📊 SECTION C: CLASS 3 (MIMICRY)")
    print("-" * 70)
    print(f"  Mimicry Samples Total:      {final_metrics['mimicry_samples']}")
    print(f"    Human (Real Authors):     {final_metrics['mimicry_human_samples']}")
    print(f"    AI (Mimicking):           {final_metrics['mimicry_ai_samples']}")
    print(f"  Mimicry Accuracy:           {final_metrics['mimicry_accuracy']:.4f}")
    print(f"  Mimicry AI Detection Rate:  {final_metrics['mimicry_detection_rate']:.4f}")
    print(f"  Human False Positive Rate:  {final_metrics['human_false_positive_rate']:.4f}")

print("\n" + "=" * 70)
print("✅ EVALUATION COMPLETE")
print("=" * 70)

## 18. Save Model and Results

In [ ]:
print("=" * 70)
print("SAVING MODEL AND RESULTS")
print("=" * 70)

# Save the LoRA adapter
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")

print(f"\n💾 LoRA adapter saved to: {OUTPUT_DIR}/lora_adapter")
print(f"\n📝 Note: Only LoRA weights are saved (~1-2MB), not the full model!")
print(f"   To load: Load base model + apply saved LoRA adapter")

# Save all evaluation metrics to CSV
metrics_df = pd.DataFrame([final_metrics])
metrics_df.to_csv(f"{OUTPUT_DIR}/final_metrics.csv", index=False)

print(f"\n💾 Final metrics saved to: {OUTPUT_DIR}/final_metrics.csv")

# Save confusion matrices
cm_data = {
    'overall_tn': [int(tn)],
    'overall_fp': [int(fp)],
    'overall_fn': [int(fn)],
    'overall_tp': [int(tp)]
}

if class3_metrics:
    cm_data['mimicry_tn'] = [class3_metrics['mimicry_tn']]
    cm_data['mimicry_fp'] = [class3_metrics['mimicry_fp']]
    cm_data['mimicry_fn'] = [class3_metrics['mimicry_fn']]
    cm_data['mimicry_tp'] = [class3_metrics['mimicry_tp']]

cm_df = pd.DataFrame(cm_data)
cm_df.to_csv(f"{OUTPUT_DIR}/confusion_matrices.csv", index=False)

print(f"💾 Confusion matrices saved to: {OUTPUT_DIR}/confusion_matrices.csv")

print("\n✅ All results saved successfully!")

SAVING MODEL

💾 LoRA adapter saved to: /home/avani/precog/reports/lora_distilbert/lora_adapter

📝 Note: Only LoRA weights are saved (~1-2MB), not the full model!
   To load: Load base model + apply saved LoRA adapter

💾 Evaluation results saved to: /home/avani/precog/reports/lora_distilbert/evaluation_results.csv


## 19. Summary

### Model Architecture:
- **Base Model**: DistilBERT (distilbert-base-uncased)
- **Fine-Tuning Method**: LoRA (Low-Rank Adaptation)
- **LoRA Configuration**: r=8, alpha=16, dropout=0.1
- **Target Modules**: Query and Value attention layers (q_lin, v_lin)
- **Task**: Binary sequence classification (Human vs AI)

### Key Achievements:
- **Parameter Efficiency**: Training only ~0.6% of parameters (LoRA adapters)
- **Memory Efficient**: Adapter weights are only ~1-2MB
- **Performance**: Comparable to full fine-tuning with 99%+ fewer trainable parameters
- **Comprehensive Evaluation**: 
  - Overall test set performance
  - Generic AI detection (Class 2)
  - Mimicry detection with BALANCED human+AI comparison (Class 3 - FIXED)
  - Error analysis with example misclassifications

### Critical Bug Fixed:
The original Class 3 evaluation only tested AI mimicry samples (support=0 for Human class), making 100% accuracy misleading. The fixed version includes BOTH:
- Real human texts from Doyle/Stevenson
- AI texts mimicking Doyle/Stevenson

This provides a true test: **Can the model distinguish real authors from AI mimicking them?**

### Why "The Transformer"?
This model leverages **transformer architecture** with **self-attention**:
- Captures long-range dependencies in text
- Learns contextual relationships between words
- Pre-trained on massive corpora, fine-tuned for stylometry
- LoRA enables efficient adaptation to new tasks

### LoRA Benefits:
1. **Efficiency**: 99%+ reduction in trainable parameters
2. **Speed**: Faster training and lower memory usage
3. **Modularity**: Easy to swap/combine multiple LoRA adapters
4. **Performance**: Maintains accuracy while being parameter-efficient
5. **Deployment**: Tiny adapter files for easy distribution

### Comparison with Other Tiers:
- **Tier A (XGBoost)**: Hand-crafted features, interpretable, fast baseline
- **Tier B (GloVe + FFN)**: Static semantic embeddings, simpler architecture
- **Tier C (DistilBERT + LoRA)**: Contextualized embeddings, self-attention, SOTA
- DistilBERT learns **contextualized representations** (word meaning depends on context)
- GloVe provides **static embeddings** (same vector for word regardless of context)

### Evaluation Metrics Tracked:
**Overall:**
- Accuracy, Precision, Recall, F1-Score (per class)
- Specificity, Sensitivity
- False Positive Rate (Human→AI)
- False Negative Rate (AI→Human)
- Confusion Matrix

**Class 2 (Generic AI):**
- Detection accuracy on non-mimicry AI texts

**Class 3 (Mimicry - FIXED):**
- Balanced evaluation on real authors + AI mimics
- Mimicry AI detection rate
- Human false positive rate
- Separate confusion matrix for mimicry subset

**Error Analysis:**
- False negatives: AI texts that fooled the model
- False positives: Real texts wrongly flagged as AI
- Confidence scores and text previews

### Output Files:
- `lora_adapter/` - LoRA weights and tokenizer (~1-2MB)
- `final_metrics.csv` - Complete metrics dictionary
- `confusion_matrices.csv` - All confusion matrices
- `checkpoint-*/` - Training checkpoints